## 1. Setup and Data Loading
Mounting Google Drive and importing required libraries.

In [ ]:
import glob
import re
import warnings

import pandas as pd
from google.colab import drive

# Suppress pandas FutureWarnings for clean output
warnings.simplefilter(action='ignore', category=FutureWarning)

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.5. Local Data Caching
Copying dataset to local Colab SSD to eliminate Google Drive I/O bottlenecks.

In [ ]:
print("Copying data to local SSD...")
!cp -r /content/drive/MyDrive/data /content/local_data
print("Copy complete!")

Copying data to local SSD...
Copy complete!


## 2. Load Parquet Files and Normalize Identifiers
Reading all processed files from DOU and Djinni, and creating a unified global ID.

In [ ]:
DATA_PATH = '/content/local_data'

def load_and_prepare_data(base_path):
    all_chunks = []
    files = [
        f for f in glob.glob(f"{base_path}/**/*.parquet", recursive=True)
        if 'dou' in f or 'djinni' in f
    ]

    for f in files:
        df = pd.read_parquet(f)
        path_lower = f.lower()

        if 'dou' in path_lower:
            source = 'dou'
        elif 'djinni' in path_lower:
            source = 'djinni'
            match = re.search(r'date=(\d{4}-\d{2}-\d{2})', f)
            if match:
                df['date'] = match.group(1)
        else:
            source = 'unknown'

        df['source'] = source
        df['global_id'] = df['source'] + "_" + df['id'].astype(str)
        df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.normalize()
        all_chunks.append(df)

    return pd.concat(all_chunks, ignore_index=True)

print("Loading data from local SSD...")
df_raw = load_and_prepare_data(DATA_PATH)
print(f"Total records loaded: {len(df_raw)}")

Loading data from local SSD...
Total records loaded: 69949


## 3. Exact ID Deduplication and Date Aggregation
Aggregating exact `global_id` matches into single records. Data is sorted newest-first (`ascending=False`). Retains the most recent text description (`first`) and captures the most recent non-null salary. All historical occurrence dates are collected into an array (`seen_dates`).

In [ ]:
print("Performing lossless ID deduplication...")

df_sorted = df_raw.sort_values(by='date', ascending=False)

def first_valid(series):
    valid_data = series.dropna()
    return valid_data.iloc[0] if not valid_data.empty else None

agg_logic = {col: 'first' for col in df_sorted.columns if col not in ['global_id', 'date']}

if 'public_salary_min' in df_sorted.columns:
    agg_logic['public_salary_min'] = first_valid
if 'public_salary_max' in df_sorted.columns:
    agg_logic['public_salary_max'] = first_valid

agg_logic['date'] = lambda x: sorted(list(x.dt.strftime('%Y-%m-%d').dropna().unique()))

df_exact_dedup = df_sorted.groupby('global_id').agg(agg_logic).reset_index()
df_exact_dedup = df_exact_dedup.rename(columns={'date': 'seen_dates'})
df_exact_dedup['repost_count'] = df_exact_dedup['seen_dates'].apply(len)

print(f"Unique IDs: {len(df_exact_dedup)} | Total rows processed: {len(df_raw)}")
df_exact_dedup[['global_id', 'title', 'repost_count', 'seen_dates']].head(5)

Performing lossless ID deduplication...
Unique IDs: 46227 | Total rows processed: 69949


,global_id,title,repost_count,seen_dates
0,djinni_102411,Data Scientist,8,"[2025-08-12, 2025-08-19, 2025-08-26, 2025-09-0..."
1,djinni_103145,Програміст 1С,1,[2026-03-16]
2,djinni_103334,Busines Analyst,1,[2026-04-15]
3,djinni_104851,Senior Backend Developer,1,[2025-03-25]
4,djinni_113154,Python developer,1,[2026-04-01]


## 4. Semantic Parent-Child Linkage
Applying NLP embeddings to identify semantically identical job descriptions across different IDs or platforms. Constructs a lossless Parent-Child hierarchy where `is_parent=True` designates the most recent master record.

In [ ]:
!pip install -q sentence-transformers
import torch
from sentence_transformers import SentenceTransformer, util

print("Initializing NLP Model (GPU)...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

df_exact_dedup['description'] = df_exact_dedup['description'].fillna("")
descriptions = df_exact_dedup['description'].tolist()

embeddings = model.encode(descriptions, batch_size=64, show_progress_bar=True, convert_to_tensor=True)
clusters = util.community_detection(embeddings, min_community_size=2, threshold=0.95)

df_exact_dedup['is_parent'] = True
df_exact_dedup['parent_id'] = None
df_exact_dedup['latest_date'] = df_exact_dedup['seen_dates'].apply(lambda x: x[-1] if x else "")

for cluster in clusters:
    cluster_df = df_exact_dedup.iloc[list(cluster)].sort_values(by='latest_date')
    parent_idx = cluster_df.index[-1]
    parent_id = df_exact_dedup.at[parent_idx, 'global_id']

    for idx in cluster:
        if idx != parent_idx:
            df_exact_dedup.at[idx, 'is_parent'] = False
            df_exact_dedup.at[idx, 'parent_id'] = parent_id

df_exact_dedup = df_exact_dedup.drop(columns=['latest_date'])

print(f"Parents (LLM targets): {df_exact_dedup['is_parent'].sum()} | Children: {(~df_exact_dedup['is_parent']).sum()}")

FINAL_PATH = '/content/drive/MyDrive/data/step2_master_vacancies_clustered.parquet'
df_exact_dedup.to_parquet(FINAL_PATH, index=False)

Initializing NLP Model (GPU)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/723 [00:00<?, ?it/s]

Parents (LLM targets): 27720 | Children: 18507
